# SmartDs Spherical Shell Mass Flux (Starter)

This notebook loads the 3D BATSRUS sample file, samples a spherical shell at `R=10`, computes the radial mass flux on that shell, and plots it as a 2D map.

It reuses the **library** for shell sampling and mass-loss integration (no VTK/PyVista, no custom resampling).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from starwinds_analysis.smart_ds import SmartDs
from starwinds_analysis.data.samples import get_sample
from starwinds_analysis.analysis.shells import (
    integrate_shell_scalar,
    sample_spherical_shells,
    sample_spherical_shells_fibonacci,
)
from starwinds_analysis.physics.flux_density import radial_advective_flux_density
from starwinds_analysis.recipes.spherical import spherical_vector_components

plt.rcParams['figure.dpi'] = 120


## Load the 3D Sample File

Defaults to `sample_data/3d__var_3_n00060000.plt`.

In [ ]:
DATA_FILE = get_sample('3d__var_1_n00060000.plt')
STAR_RADIUS_M = 696_000_000.0  # feed into griblet/BATSRUS graph near the top
sds = SmartDs.from_file(DATA_FILE)
sds.add_batsrus_graph(body_radius_m=STAR_RADIUS_M)
# DONE how many times do i need to say to remove this? just put print(sds).
print(sds)


## Sample a Spherical Shell at `R = 10` and Compute Mass Flux

The shell sampling uses the library grid sampler (for plotting on a regular `theta/phi` grid).

In [ ]:
R_SHELL = 10.0  # in BATSRUS coordinate units (typically body radii)
N_POLAR = 48
N_AZIMUTH = 2 * N_POLAR

shell = sample_spherical_shells(
    sds,
    [R_SHELL],
    fields=('Rho [kg/m^3]', 'U_x [m/s]', 'U_y [m/s]', 'U_z [m/s]'),
    n_polar=N_POLAR,
    n_azimuth=N_AZIMUTH,
    length_unit_to_m=STAR_RADIUS_M,
    method='nearest',
)

rho_shell = shell('Rho [kg/m^3]')[0]
ux_shell = shell('U_x [m/s]')[0]
uy_shell = shell('U_y [m/s]')[0]
uz_shell = shell('U_z [m/s]')[0]
u_r_shell, _u_theta_shell, _u_phi_shell = spherical_vector_components(
    ux_shell, uy_shell, uz_shell, shell('X [R]')[0], shell('Y [R]')[0], shell('Z [R]')[0]
)
mass_flux_shell = radial_advective_flux_density(rho_shell, u_r_shell)

lon_deg = np.degrees(shell('phi [rad]')[0])
lat_deg = 90.0 - np.degrees(shell('theta [rad]')[0])

print('Shell grid shape (theta, phi):', mass_flux_shell.shape)
{
    'finite_cells': int(np.count_nonzero(np.isfinite(mass_flux_shell))),
    'total_cells': int(mass_flux_shell.size),
    'nonpositive_cells': int(np.count_nonzero(np.isfinite(mass_flux_shell) & (mass_flux_shell <= 0.0))),
    'min [kg/m^2/s]': float(np.nanmin(mass_flux_shell)),
    'max [kg/m^2/s]': float(np.nanmax(mass_flux_shell)),
}


In [ ]:
# The map is already in SI units (kg/m^2/s) on a native lon/lat shell grid.
# DONE for scripts that take time, we should generate the data in a different cell than in the plotting cell.
# Cell 5 computes the shell data; the plotting happens below.


In [ ]:
# DONE have a linear scale option easily avaiable.
MASS_FLUX_SCALE = 'log'  # change to 'linear' for a linear color scale


In [ ]:
# 2D shell plot in longitude/latitude using the native theta/phi sampling grid
# We assume the wind mass flux is mostly outward (>0). Non-positive values are shown with the colormap under-color.
# DONE the percentiles scaling is too fancy. We can just use the min/max of the positive values.
fig, ax = plt.subplots(figsize=(9, 4.5), constrained_layout=True)

if MASS_FLUX_SCALE == 'log':
    positive = np.isfinite(mass_flux_shell) & (mass_flux_shell > 0.0)
    if np.any(positive):
        positive_min = float(mass_flux_shell[positive].min())
        mass_flux_plot = np.array(mass_flux_shell, copy=True)
        mass_flux_plot[~positive] = 0.1 * positive_min
        img = ax.pcolormesh(
            lon_deg, lat_deg, mass_flux_plot, shading='nearest', cmap='viridis',
            norm='log', vmin=positive_min,
        )
        img.cmap.set_under('magenta')
        print(f'Under-color cells shown: {int(np.count_nonzero(~positive))}')
    else:
        img = ax.pcolormesh(lon_deg, lat_deg, mass_flux_shell, shading='nearest', cmap='viridis')
        print('No positive shell mass-flux cells found; using linear coloring for this plot.')
else:
    img = ax.pcolormesh(lon_deg, lat_deg, mass_flux_shell, shading='nearest', cmap='viridis')

ax.set_xlabel('Longitude [deg]')
ax.set_ylabel('Latitude [deg]')
ax.set_title(f'Mass flux on shell at R={R_SHELL:g}')
ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
fig.colorbar(img, ax=ax, label="Mass flux [kg/m^2/s]")
plt.show()


## Mass-Loss Rate at `R = 10` (Library Function)

This uses `mass_loss_vs_radius(...)` with **Fibonacci-sphere sampling** for the shell integral.
For notebook responsiveness we use a lower integration resolution than the plotting shell.
(The runtime scales with the number of shell sample points.)

In [ ]:
# Optional cross-check: integrate the plotted grid-shell mass flux directly
mass_loss_grid_arr, coverage_grid_arr = integrate_shell_scalar(mass_flux_shell[None, ...], shell('dA [m^2]')[:1])
mass_loss_grid = float(mass_loss_grid_arr[0])
coverage_grid = float(coverage_grid_arr[0])

# Use a smaller Fibonacci shell for the integral so this cell stays responsive.
# Effective Fibonacci sample count is N_MASS_LOSS_POLAR * N_MASS_LOSS_AZIMUTH.
N_MASS_LOSS_POLAR = 16
N_MASS_LOSS_AZIMUTH = 32
N_MASS_LOSS_POINTS = N_MASS_LOSS_POLAR * N_MASS_LOSS_AZIMUTH
print(f'Fibonacci integration points: {N_MASS_LOSS_POINTS}')

# DONE Why do you have this dumbass function? What do you think the notebooks are for?
shell_fib = sample_spherical_shells_fibonacci(
    sds,
    [R_SHELL],
    fields=('Rho [kg/m^3]', 'U_x [m/s]', 'U_y [m/s]', 'U_z [m/s]'),
    n_points=N_MASS_LOSS_POINTS,
    length_unit_to_m=STAR_RADIUS_M,
    method='nearest',
)

rho_fib = shell_fib('Rho [kg/m^3]')
ux_fib = shell_fib('U_x [m/s]')
uy_fib = shell_fib('U_y [m/s]')
uz_fib = shell_fib('U_z [m/s]')
u_r_fib, _u_theta_fib, _u_phi_fib = spherical_vector_components(ux_fib, uy_fib, uz_fib, shell_fib('X [R]'), shell_fib('Y [R]'), shell_fib('Z [R]'))
mass_flux_fib = radial_advective_flux_density(rho_fib, u_r_fib)
mass_loss_fib_arr, coverage_fib_arr = integrate_shell_scalar(mass_flux_fib, shell_fib('dA [m^2]'))
dotm_fib = float(mass_loss_fib_arr[0])
cov_fib = float(coverage_fib_arr[0])

print(f'Grid-shell integral (same plotted shell): {mass_loss_grid:.6e} kg/s')
print(f'Grid-shell coverage             : {coverage_grid:.6f}')
print(f'Fibonacci-shell integral        : {dotm_fib:.6e} kg/s')
print(f'Fibonacci-shell coverage        : {cov_fib:.6f}')
